In [ ]:

import json
import re
import os
import static_ana
import javalang

from pathlib import Path
from typing import List, Dict, Set, Callable
import string

from openai import OpenAI

DEFAULT_LLM = "qwen3-32b"
API_KEY = "YOUR_API_KEY"
BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"

LOG_METHODS = ["error", "fatal", "info", "warn", "debug"]

In [ ]:
class LLMWrapper:
    def __init__(self, api_key=API_KEY, model=DEFAULT_LLM, tools=None, base_url=BASE_URL):
        self.client = OpenAI(
            api_key=api_key,
            base_url=base_url,
        )
        self.model = model
        self.tools = tools
        self.context = []

    # Only streaming calls support thinking
    def call_stream(self, prompt, role="user", enable_tools=False, temperature=0.1):
        if not self.context:
            self.context.append({"role": "system", "content": "You need to strictly follow the output format required by the user, and not output anything outside of the JSON object. Answer in English"})
        try:
            if prompt:
                self.context.append({"role": role, "content": prompt, "tools": self.tools if enable_tools else None})
            else:
                self.context[-1]["tools"] = self.tools if enable_tools else None
            response = self.client.chat.completions.create(
                model=self.model, # Model list: https://help.aliyun.com/zh/model-studio/getting-started/models
                # Add /think or /no_think in the latest prompt to control COT
                extra_body={"enable_thinking": False},
                stream=True,
                messages=self.context,
                tools=self.tools if enable_tools else None,
                temperature=temperature,
                max_tokens=8192,
                response_format={"type": "json_object"}
            )
            reasoning_content = ""
            answer_content = ""
            tool_info = []
         
            for chunk in response:
                if not chunk.choices:
                    return ""
                else:
                    delta = chunk.choices[0].delta
                    # think process
                    if hasattr(delta, 'reasoning_content') and delta.reasoning_content is not None:
                        reasoning_content += delta.reasoning_content
                        
                    # answer process
                    else:
                        if delta.content is not None:
                            answer_content += delta.content
                        
                        # tool call
                        if delta.tool_calls is not None:
                            for tool_call in delta.tool_calls:
                                index = tool_call.index  # Tool call index, for parallel calls
                                
                                # Dynamically extend tool info storage list
                                while len(tool_info) <= index:
                                    tool_info.append({})
                                
                                # Collect tool call ID
                                if tool_call.id:
                                    tool_info[index]['id'] = tool_info[index].get('id', '') + tool_call.id
                                
                                # Collect function name
                                if tool_call.function and tool_call.function.name:
                                    tool_info[index]['name'] = tool_info[index].get('name', '') + tool_call.function.name
                                
                                # Collect function arguments (json str)
                                if tool_call.function and tool_call.function.arguments:
                                    tool_info[index]['arguments'] = tool_info[index].get('arguments', '') + tool_call.function.arguments
             
            content  = reasoning_content+" "+answer_content
            ret = answer_content
            return ret
        except: 
            raise
    
    def clear_context(self):
        self.context = []    
        
    def set_context(self, context):
        self.context = context

llm = LLMWrapper()
llm.call_stream("hello")

In [ ]:
def is_log_method(s):
    return s.lower() in [method.lower() for method in LOG_METHODS]

def build_static_analysis_context(java_code: str, file_path: Path, class_to_file_map: Dict) -> List[Dict]:
    """
    Perform static analysis on a single Java file, extract all log calls and their cross-file call chains, and return structured results for LLM use.
    """
    try:
        log_calls = static_ana.extract_log_calls(java_code)
        if not log_calls:
            return []
        
        tree = javalang.parse.parse(java_code)
        package, import_map = static_ana.parse_imports_and_package_v2(java_code)
        simple_class_name = file_path.stem
        current_class_full = f"{package}.{simple_class_name}" if package else simple_class_name

        analysis_results = []

        for log_call in log_calls:
            if not is_log_method(log_call['method']):
                continue

            all_paths = static_ana.trace_all_log_call_paths_across_files(java_code, log_call, class_to_file_map)
            path_templates = []

            for path in all_paths:
                segments = []
                for cls, invocation, stmt_or_info in path:
                    if cls == "UnknownClass":
                        cls = current_class_full  # Use current file full class name
                    if "Log method call" in stmt_or_info:
                        continue  # Starting point, ignore
                    elif "✅ Built-in method" in stmt_or_info or "❓ Unrecognized function" in stmt_or_info or "🔄 Already visited" in stmt_or_info:
                        segments.append("<.*>")
                    else:
                        # Try to extract string literals from return statements
                        # Example: return "User_" + uid.toUpperCase() + "_NotFound";
                        # Should extract → ["User_", <.*>, "_NotFound"]
                        # Match all double-quoted strings
                        parts = []
                        remaining = stmt_or_info

                        while remaining:
                            # Find the first string literal
                            str_match = re.search(r'\"([^\"]*)\"', remaining)
                            if not str_match:
                                # No more strings, treat the rest as variables
                                if remaining.strip() and not remaining.strip().startswith("return"):
                                    parts.append("<.*>")
                                break

                            # Add the non-string part before the string (treated as variable/expression)
                            pre_str = remaining[:str_match.start()]
                            if pre_str.strip() and not pre_str.strip().startswith("return"):
                                parts.append("<.*>")

                            # Add string literal
                            parts.append(str_match.group(1))

                            # Intercept the rest
                            remaining = remaining[str_match.end():]

                        # If there is still non-empty content at the end, treat it as a variable
                        if remaining.strip() and not remaining.strip().startswith("return"):
                            parts.append("<.*>")

                        # Merge the extraction results of this layer
                        segments.extend(parts if parts else ["<.*>"])

                # Splice the template for this path
                path_template = "".join(segments) if segments else "<.*>"
                path_templates.append(path_template)

            analysis_results.append({
                "line": log_call["line"],
                "method": log_call["method"],
                "initial_template": log_call["template"],
                "call_paths": all_paths,
                "derived_templates": path_templates
            })

        return analysis_results

    except Exception as e:
        print(f"❌ Static analysis failed {file_path}: {e}")
        return []

def format_log_call_analysis_for_llm(java_code: str, file_path: Path, class_to_file_map: Dict) -> str:
    """
    Simulate the output format of test_full_cross_file_analysis to generate human-readable static analysis reports for LLM use.
    Only include ERROR and FATAL level log calls.
    """
    try:
        log_calls = static_ana.extract_log_calls(java_code)
        if not log_calls:
            return "❌ No log calls extracted"

        tree = javalang.parse.parse(java_code)
        package, import_map = static_ana.parse_imports_and_package_v2(java_code)
        simple_class_name = file_path.stem
        current_class_full = f"{package}.{simple_class_name}" if package else simple_class_name

        output_lines = []
        output_lines.append(f"🔍 Extracted {len(log_calls)} log call(s)")

        for idx, log_call in enumerate(log_calls):
            if not is_log_method(log_call['method']):
                continue

            output_lines.append(f"\n=== Analyzing log call #{idx + 1} ===")
            output_lines.append(f"Location: line {log_call['line']}, method: {log_call['method']}")
            output_lines.append(f"Template: {log_call['template']}")

            all_paths = static_ana.trace_all_log_call_paths_across_files(java_code, log_call, class_to_file_map)
            output_lines.append(f"\n=== 🌐 Call path analysis results (total {len(all_paths)} paths) ===")

            for i, path in enumerate(all_paths, 1):
                output_lines.append(f"\n--- Path {i} ---")
                for j, (cls, invoc, stmt) in enumerate(path):
                    # Fix: If the first layer is UnknownClass, replace it with the current class name
                    if j == 0 and cls == "UnknownClass":
                        cls = current_class_full
                    output_lines.append(f"  {j+1}. Class: {cls}")
                    output_lines.append(f"     Invocation code: {invoc}")
                    output_lines.append(f"     Called function info: {stmt}")

        return "\n".join(output_lines)

    except Exception as e:
        return f"❌ Static analysis failed: {e}"

def create_extraction_prompt(java_code, static_analysis_report, log_methods, with_static_ana):
    """
    Args:
        java_code (str): Java Source code
        static_analysis_report (str): Static analysis report（Including full function source code）
        log_methods (List[str]): List of log method names to extract，e.g. ["error", "fatal", "info"]
    """
    # Convert method name to uppercase for description
    levels_str = ", ".join(f'"{m.upper()}"' for m in log_methods)
    methods_str = ", ".join(f'`{m}`' for m in log_methods)

    if with_static_ana:
        return f"""
    You are an advanced Java log template extractor. Please combine the provided source code and the human-readable static analysis report to extract all log patterns for the specified levels.

    # ⚠️ Important Premise:
    - **Static analysis reports may be incomplete**! It relies on AST parsing and may miss log calls due to:
        - File syntax errors (unable to parse)
        - Log calls located in comments or string literals (false positives/negatives)
        - Use of dynamic method calls (e.g., reflection), complex nesting of Lambda expressions, etc.
    - **You must use the original Java source code as the authoritative source** and comprehensively scan for all log calls.
    - Static analysis reports are only used to **assist in understanding the call chain and function body logic** and **cannot replace source code scanning**.

    # Log methods to process:
    The following log methods need to be processed: {methods_str}  
    The `level` field in the corresponding output should be in uppercase: {levels_str}

    # Input Format:
    1. Java source code (located between [[[JAVA CODE BEGIN]]] and [[[JAVA CODE END]]])
    2. Static analysis report (human-readable text), including:
    - Position, method name, and initial template of each log call
    - Each call path (cross-method/cross-file), showing each layer:
            - Class: Class name (e.g., com.example.A)
            - Invocation code: Method invocation statement
            - Called function info: **Complete function definition source code** (including method signature, method body, local variables, control flow, all return statements)

    # Key Background: Invocation rules of Java logging frameworks (e.g., SLF4J)
    - Log methods are usually: `logger.error(String format, Object... arguments)`
    - The number of `{{}}` placeholders in the format string determines **how many subsequent arguments participate in formatting**.
    - **Extra arguments (e.g., Throwable exception objects) will be directly appended to the end of the final log message but are not part of the "template" content**.
    - Example：
        logger.error("Code={{}}", code, e);  
        → Actual log = `"Code=123" + "\\n" + e.toString()`  
        → **But the template should only be `"Code=<.*>"`, and the exception `e` does not generate an extra `<.*>`**

    # Your Task:
    - **Process all log calls belonging to [{methods_str}]**.
    - **Infer templates based only on the format string (i.e., the first argument of the log call) and its call chain**.
    - **Ignore extra arguments in log calls that exceed the number of `{{}}` (especially exception `e`), they do not contribute `<.*>` to the template**.
    - **Make full use of function source code context** for precise string value flow analysis:
        - ✅ **Constants**: String literals, or local variables explicitly assigned as literals and not modified;
        - ❌ **Variables**: Method arguments, fields, results of operations on variables, unknown function return values.
    - **Use `<.*>` as a substitute only in the following cases**:
        a) Expression contains variables of uncertain origin (e.g., argument `uid`);
        b) Any operation performed on variables (e.g., `.toUpperCase()`, `+ id`);
        c) Call to an unrecognized function (❓) or built-in method (✅);
        d) Each `{{}}` placeholder in the original format string (must be replaced one-to-one with `<.*>`);
        e) Values in control flow branches that cannot be statically determined.
    - **The `level` field in the output must be the uppercase form of the log method name** (e.g., if the method is `warn` → `"level": "WARN"`).

    # Output Requirements:
    - Output must be a JSON array, with each element in the format: {{"method": "FullClassName.MethodName", "pattern": "SplicedTemplate", "level": "LEVEL_NAME"}}
    - For example：{{"method": "com.example.A.error", "pattern": "xxx<.*>yyy", "level": "ERROR"}}

    # Example Descriptions:

    ## Example 1: warn call with exception
    Source code:
        logger.warn("User {{}} not found", userId, e);
    → Method `warn` is in the processing list →  
    → Template: `"User <.*> not found"`, level: `"WARN"`

    ## Example 2: info level without placeholders
    Source code:
        LOG.info("System started");
    → Template：`"System started"`，level: `"INFO"`

    ## Example 3: debug level + variable concatenation
    Called function returns `"Debug: " + value`, call: log.debug(msg);
    → Template: `"Debug: <.*>"`, level: `"DEBUG"`

    ## Example 4: Local constants
    public static String getTag() {{ return "[TRACE]"; }}
    Call: log.trace(getTag() + " entry");
    → Template: `"[TRACE] entry"`, level: `"TRACE"`

    # Important Rules:
    - **Strictly maintain the original structure** when concatenating, do not add spaces between `<.*>` and strings;
    - Punctuation, spaces, parentheses, etc., in string literals must be preserved as is;
    - Each path outputs a template independently；
    - **Never add `<.*>` due to exceptions or other additional arguments**;
    - **Only process methods in {methods_str}, ignore other log calls**.

    Now please process the following input:

    ===
    [[[JAVA CODE BEGIN]]]
    {java_code}
    [[[JAVA CODE END]]]

    ===
    Static analysis report:
    {static_analysis_report}

    ===
    Please output a JSON array that meets the requirements (do not include any explanation, comments, or extra text):
    """ 
    else:
        # ========== New Prompt (Only includes Java Source code) ==========
        return f"""
You are an advanced Java log template extractor. Please extract all log patterns for the specified levels based only on the provided Java source code.

# Log methods to process:
The following log methods need to be processed: {methods_str}  
The `level` field in the corresponding output should be in uppercase: {levels_str}

# Input Format:
- Java source code (located between [[[JAVA CODE BEGIN]]] and [[[JAVA CODE END]]])

# Key Background: Invocation rules of Java logging frameworks (e.g., SLF4J)
- Log methods are usually: `logger.error(String format, Object... arguments)`
- The number of `{{}}` placeholders in the format string determines **how many subsequent arguments participate in formatting**.
- **Extra arguments (e.g., Throwable exception objects) will be directly appended to the end of the final log message but are not part of the "template" content**.
- Example：
    logger.error("Code={{}}", code, e);  
    → Actual log = `"Code=123" + "\\n" + e.toString()`  
    → **But the template should only be `"Code=<.*>"`, and the exception `e` does not generate an extra `<.*>`**

# Your Task:
- **Process all log calls belonging to [{methods_str}]**.
- **Infer the template based only on the first argument (format string) of the log call**.
- **Ignore extra arguments in log calls that exceed the number of `{{}}` (especially exception `e`), they do not contribute `<.*>` to the template**.
- **Perform string value flow analysis**：
    - If the format string is a **direct string literal** (e.g., `"Error occurred"`), the template is that literal.
    - If the format string is a **variable or expression** (e.g., `errorMsg`, `prefix + " failed"`), the result of the entire expression should be treated as a `<.*>`.
    - If the format string contains **concatenation of string literals and variables** (e.g., `"User " + userId + " not found"`), you need to keep the literal parts and replace the variable parts with `<.*>`. For example, the template for the above example should be `"User <.*> not found"`.
- **The `level` field in the output must be the uppercase form of the log method name** (e.g., if the method is `warn` → `"level": "WARN"`).

# Output Requirements:
- Output must be a JSON array, with each element in the format: {{"method": "FullClassName.MethodName", "pattern": "SplicedTemplate", "level": "LEVEL_NAME"}}
  - For example：{{"method": "com.example.A.error", "pattern": "xxx<.*>yyy", "level": "ERROR"}}

# Example Descriptions:

## Example 1: warn call with exception
Source code:
    logger.warn("User {{}} not found", userId, e);
→ Method `warn` is in the processing list →  
→ Template: `"User <.*> not found"`, level: `"WARN"`

## Example 2: info level without placeholders
Source code:
    LOG.info("System started");
→ Template：`"System started"`，level: `"INFO"`

## Example 3: Format string is a variable
Source code:
    String msg = "Debug: " + value;
    log.debug(msg);
→ Template: `"Debug: <.*>"`, level: `"DEBUG"`

## Example 4: Format string is a constant function call
Source code:
    public static String getTag() {{ return "[TRACE]"; }}
    ...
    log.trace(getTag() + " entry");
→ **Note**: Since there is no call chain information, the LLM needs to judge for itself whether `getTag()` is a constant.
→ The safest assumption is to treat it as a variable, so the template may be `"<.*> entry"`.
→ (This is a key point of the ablation experiment: lack of context leads to a decrease in precision)

# Important Rules:
- **Strictly maintain the original structure** when concatenating, do not add spaces between `<.*>` and strings;
- Punctuation, spaces, parentheses, etc., in string literals must be preserved as is;
- **Never add `<.*>` due to exceptions or other additional arguments**;
- **Only process methods in {methods_str}, ignore other log calls**.

Now please process the following input:

===
[[[JAVA CODE BEGIN]]]
{java_code}
[[[JAVA CODE END]]]

===
Please output a JSON array that meets the requirements (do not include any explanation, comments, or extra text):
"""


def extract_log_patterns(java_dir: str, out_dir: str, log_levels=LOG_METHODS, with_static_ana=True, model=DEFAULT_LLM):
    """
    Recursively traverse Java source files in the specified directory and use LLM to extract all ERROR and FATAL level log templates.
    Args:
        java_dir (str): Root directory path containing Java source code.
        out_dir (str): Result directory
    """
    # 1. Initialization
    all_patterns: Set[str] = set()
    result_list: List[Dict] = []

    # Build global class name to file path mapping (for cross-file analysis)
    class_to_file_map = static_ana.build_class_to_file_map(java_dir)

    # Create LLM client instance
    llm = LLMWrapper(model=model)

    java_dir_path = Path(java_dir)
    if not java_dir_path.exists():
        raise FileNotFoundError(f"Specified directory does not exist: {java_dir}")

    bad_case = []
    for file_path in java_dir_path.rglob("*"):
        if (file_path.is_file()
            and file_path.suffix == '.java'
            and 'test' not in file_path.parts):

            print(f"Processing file: {file_path}")

            with open(file_path, 'r', encoding='utf-8') as f:
                java_code = f.read()

            # Execute static analysis，Generate human-readable report（FormatConsistent with test function）
            static_analysis_report = format_log_call_analysis_for_llm(java_code, file_path, class_to_file_map)
            #print(static_analysis_report)
            # Construct full prompt (note double curly brace escaping)
            full_prompt = create_extraction_prompt(
                java_code=java_code,
                static_analysis_report=static_analysis_report,
                log_methods = log_levels,
                with_static_ana=with_static_ana
            )
            #print(full_prompt)
            try:
                llm.clear_context()
                llm_response = llm.call_stream(full_prompt)
            except Exception as e:
                print(f"LLM call error: {e}")
                llm_response = "REQUEST_LLM_FAILED"
                bad_case.append({"file": str(file_path), "llm_resp": llm_response})
                continue

            try:
                patterns_from_file: List[Dict] = json.loads(llm_response)
                assert isinstance(patterns_from_file, list)
                for pattern_dict in patterns_from_file:
                    assert "method" in pattern_dict and "pattern" in pattern_dict and "level" in pattern_dict
                    # Normalize level field (case compatible)
                    level = pattern_dict["level"].upper()
                    allowed_levels = {lvl.upper() for lvl in log_levels}
                    if level not in allowed_levels:
                        continue
                    pattern_dict["level"] = level
                    pattern_str = json.dumps(pattern_dict, sort_keys=True)
                    all_patterns.add(pattern_str)
            except Exception as e:
                print(f"Error parsing JSON returned by LLM: {e}")
                print(f"Original LLM response: {llm_response}")
                bad_case.append({"file": str(file_path), "llm_resp": llm_response})
                continue

    # 👇 After all files are processed, output results 👇

    for pattern_str in all_patterns:
        try:
            pattern_dict = json.loads(pattern_str)
            result_list.append(pattern_dict)
        except json.JSONDecodeError:
            continue

    try:
        pattern_file_path = Path(os.path.join(out_dir, "pattern.json"))
        pattern_file_path.parent.mkdir(parents=True, exist_ok=True)
        with open(pattern_file_path, 'w', encoding='utf-8') as f:
            json.dump(result_list, f, indent=2, ensure_ascii=False)
        with open(os.path.join(out_dir, "bad_cases.json"), 'w', encoding='utf-8') as f:
            json.dump(bad_case, f, indent=2, ensure_ascii=False)
        print(f"Extracted {len(result_list)} patterns, saved to {pattern_file_path}")
    except Exception as e:
        print(f"Error occurred while saving results to file {pattern_file_path}: {e}")



In [ ]:
def post_process(input_dir: str, log_levels, model=DEFAULT_LLM):
    """
    Post-processing function: retry bad cases + clean pattern + output processed_pattern.json
    Use extensible rule list for pattern filtering and respect the passed log_levels parameter.
    """
    # Normalize log_levels: ensure it's a set of uppercase strings
    if not isinstance(log_levels, (list, tuple)):
        raise ValueError("log_levels must be a list or tuple of strings")
    allowed_levels = {str(level).upper() for level in log_levels}

    input_path = Path(input_dir)
    pattern_file = input_path / "pattern.json"
    bad_cases_file = input_path / "bad_cases.json"
    output_file = input_path / "processed_pattern.json"

    # Step 1: Load original pattern.json
    if not pattern_file.exists():
        print(f"❌ {pattern_file} does not exist")
        all_patterns = []
    else:
        with open(pattern_file, 'r', encoding='utf-8') as f:
            try:
                all_patterns = json.load(f)
                if not isinstance(all_patterns, list):
                    all_patterns = []
            except json.JSONDecodeError:
                print(f"❌ {pattern_file} JSON parsing failed")
                all_patterns = []

    # Step 2: Retry bad_cases.json
    if bad_cases_file.exists():
        with open(bad_cases_file, 'r', encoding='utf-8') as f:
            try:
                bad_cases = json.load(f)
                assert isinstance(bad_cases, list)
            except (json.JSONDecodeError, AssertionError):
                print(f"❌ {bad_cases_file} JSON parsing failed or format error")
                bad_cases = []
    else:
        bad_cases = []

    llm = LLMWrapper(model=model)  # Assuming LLMWrapper is defined

    for case in bad_cases:
        file_path_str = case.get("file")
        llm_resp = case.get("llm_resp", "")
        if not file_path_str:
            continue

        file_path = Path(file_path_str)
        if not file_path.exists():
            print(f"⚠️ File does not exist, skipping retry: {file_path}")
            continue

        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                java_code = f.read()
        except Exception as e:
            print(f"⚠️ Unable to read file {file_path}: {e}")
            continue

        print(f"re-prompt {file_path}")
        retry_prompt = f"""
Your previous processing results for the following Java file were unsatisfactory (obtained meaningless log templates). Please re-analyze and extract meaningful log patterns.

# Requirements:
- Only extract {', '.join(log_levels)} level logs;
- Extract **definite string literals** from log statements and call chains to splice into templates;
- All variables, function calls, expressions, and {{}} placeholders in the original log → must be replaced with <.*>;
- The template must contain at least one non-<.*> string constant (otherwise discard);
- Output JSON Array，Format：{{"method": "FullClassName.MethodName", "pattern": "xxx<.*>yyy", "level": "ERROR"}}

# Original Source Code:
[[[JAVA CODE BEGIN]]]
{java_code}
[[[JAVA CODE END]]]

# Your previous erroneous output:
{llm_resp}

# Please output the correct JSON array again (no explanation):
"""

        try:
            llm.clear_context()
            new_response = llm.call_stream(retry_prompt)
            new_patterns: List[Dict] = json.loads(new_response)
            success = False
            if isinstance(new_patterns, list):
                success = True
                for p in new_patterns:
                    if (isinstance(p, dict) and
                        "method" in p and "pattern" in p and "level" in p and
                        isinstance(p["pattern"], str)):
                        p["level"] = p["level"].upper()
                        # ✅ Key Fix: Use dynamic allowed_levels instead of hardcoding
                        if p["level"] in allowed_levels:
                            p["pattern"] = p["pattern"].replace("{}", "<.*>")
                            all_patterns.append(p)
                            
            print('statues: ', success, 'resp: ', new_response)
        except Exception as e:
            print(f"⚠️ Retry {file_path} failed: {e}")
            continue

    # ==============================
    # ✅ Define post-processing rule list
    # Each rule: Callable[[str, Dict], bool] → True=Keep, False=Discard
    # ==============================
    def rule_not_only_wildcards(pattern: str, item: Dict) -> bool:
        """Rule 1: Cannot consist only of <.*> and whitespace"""
        cleaned = re.sub(r'<\.\*>', '', pattern)
        cleaned = re.sub(r'\s+', '', cleaned)
        return len(cleaned) > 0

    def rule_contains_wildcard_or_constant(pattern: str, item: Dict) -> bool:
        """Rule 2: Must contain at least one <.*> or non-empty string (redundant protection, optional)"""
        return len(pattern.strip()) > 0

    def rule_no_empty_pattern(pattern: str, item: Dict) -> bool:
        """Rule 3: pattern cannot be empty"""
        return pattern.strip() != ""

    def rule_merge_consecutive_wildcards(pattern: str, item: Dict) -> bool:
        """Rule 4: Merge consecutive <.*> into a single <.*>"""
        merged_pattern = re.sub(r'(<\.\*>)+', '<.*>', pattern)
        item['pattern'] = merged_pattern
        return True  # Rule always returns True because we are just modifying content
    def rule_has_meaningful_content(pattern: str, item: Dict) -> bool:
        """
        Rule 5: Template must contain at least one meaningful character (letters, numbers, Chinese, etc.),
            cannot consist only of <.*>, punctuation, or spaces.
        """
        # 1. Remove all <.*> wildcards
        cleaned = re.sub(r'<\.\*>', '', pattern)
        # 2. Remove all whitespace
        cleaned = re.sub(r'\s+', '', cleaned)
        # 3. If empty, obviously invalid
        if not cleaned:
            return False
        # 4. Check if it contains at least one non-punctuation character (letters, numbers, Chinese, etc.)
        # Punctuation definition: string.punctuation + common Chinese punctuation
        punctuation = string.punctuation + '，。！？；：""''（）【】《》、'
        # If all characters in cleaned are in punctuation → meaningless
        if all(char in punctuation for char in cleaned):
            return False
        return True

    POST_PROCESS_RULES: List[Callable[[str, Dict], bool]] = [
        rule_not_only_wildcards,
        rule_no_empty_pattern,
        rule_contains_wildcard_or_constant,
        rule_merge_consecutive_wildcards,
        rule_has_meaningful_content,   
    ]

    # Step 3: Apply rule cleaning + de-duplication
    cleaned_patterns = []
    seen = set()

    for p in all_patterns:
        if not isinstance(p, dict):
            continue
        if "pattern" not in p or not isinstance(p["pattern"], str):
            continue

        pattern_str = p["pattern"]
        # Unify replace {} → <.*>
        pattern_str = pattern_str.replace("{}", "<.*>")
        p["pattern"] = pattern_str

        # Apply all rules
        should_keep = True
        for rule in POST_PROCESS_RULES:
            if not rule(p["pattern"], p):
                should_keep = False
                break

        if not should_keep:
            continue

        # De-duplicate (based on method + pattern + level)
        key = (p.get("method", ""), p["pattern"], p.get("level", ""))
        if key in seen:
            continue
        seen.add(key)

        cleaned_patterns.append(p)

    # Step 4: Save results
    try:
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(cleaned_patterns, f, indent=2, ensure_ascii=False)
        print(f"✅ Post-processing complete, total {len(cleaned_patterns)} valid patterns, saved to {output_file}")
    except Exception as e:
        print(f"❌ Saving {output_file} failed: {e}")


In [ ]:
if __name__=="__main__":
    dataset = 'example_project'
    extract_log_patterns(f"./codebases/{dataset}/", f"./log_template/{DEFAULT_LLM}/{dataset}")
    post_process(f"./log_template/{DEFAULT_LLM}/{dataset}/", log_levels=LOG_METHODS)

In [ ]:
if __name__=="__main__":
    dataset = 'Zookeeper'
    extract_log_patterns(f"./codebases/{dataset}/", f"./log_template/{DEFAULT_LLM}/{dataset}")
    post_process(f"./log_template/{DEFAULT_LLM}/{dataset}/", log_levels=LOG_METHODS)